# CyberRAG V1.3 — RAG de vulnerabilidades CISA KEV (Google Colab)
**Mini proyecto ICESI | Deep Learning Avanzado — Unidad 3: Transformers**  
**Modalidad:** corpus público real, análisis exploratorio, búsqueda léxica y semántica, generación aumentada por recuperación y demostración Gradio.  
**Backend V1.3:** Hugging Face Transformers **sin Ollama**, para evitar el bloqueo observado en Colab.  
**Reproducibilidad:** no se incluyen resultados fabricados; los resultados se generan al ejecutar.  
**Precaución:** el catálogo KEV indica explotación conocida, pero no contiene CVSS de forma sistemática ni garantiza que un producto esté afectado sin validación adicional. No introducir datos confidenciales.  
**Ejecución:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU` (recomendado). Abre **un entorno nuevo**, y selecciona `Ejecutar todas`. No ejecutes celdas de instalación de V1/V1.1.

## 0. Preparación limpia y dependencias (ejecutar UNA vez)

Esta versión utiliza **Transformers + GPU T4** (sin Ollama) y Gradio 5. Colab trae paquetes opcionales que pueden generar conflictos con sus dependencias (`hf-gradio`, `diffusers`, `google-adk`, `google-genai`, `python-fasthtml`). Como CyberRAG no utiliza esos paquetes, la celda retira esos complementos del entorno **temporal de Colab**, instala una única combinación de bibliotecas y hace una comprobación de importaciones. **No ejecutar las celdas de instalación de V1/V1.1/V1.3.**

Para reproducibilidad, recomendamos **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** antes de abrir este cuaderno; los archivos temporales se perderán. Si Colab solicita reiniciar la sesión de Python después de instalar, hazlo y continúa desde la celda 1 sin repetir instalaciones.

In [ ]:
# Ejecutar una sola vez en una sesión NUEVA de Colab.
# Retiramos únicamente complementos opcionales NO utilizados en CyberRAG.
%pip -q uninstall -y hf-gradio diffusers google-adk google-genai python-fasthtml
%pip -q install "numpy>=1.26,<3" "pandas>=2.1,<3" "matplotlib>=3.7" "requests>=2.31" "scikit-learn>=1.3,<2" "huggingface-hub>=0.34,<1" "transformers>=4.49,<5" "accelerate>=1,<2" "sentence-transformers>=3,<6" "gradio>=5.35,<6"


## 1. Verificación de entorno

Se verifica que **las librerías realmente utilizadas por el proyecto importen correctamente**; otras aplicaciones opcionales preinstaladas por Colab no se consideran parte de CyberRAG. Un aviso de pip sobre un paquete ajeno no demuestra un fallo de ejecución de este cuaderno. La prueba de importaciones y GPU es el criterio principal. **Si aquí falla un import, detente y comparte el error.**

In [ ]:
import os, re, json, time, math, random, platform
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests, torch, gradio as gr
import transformers, sentence_transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('/content/cyberrag_resultados'); OUT.mkdir(parents=True, exist_ok=True)
print('Python:', platform.python_version(), 'Torch:',torch.__version__, 'Transformers:',transformers.__version__,'Gradio:',gr.__version__)
print('CUDA disponible:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
else: print('ADVERTENCIA: sin GPU; la generación puede ser lenta.')
# Comprobación de bibliotecas esenciales, sin ejecutar inferencia todavía
from importlib.metadata import version
for pkg in ('gradio','gradio-client','transformers','huggingface-hub','sentence-transformers'):
    print(pkg,version(pkg))
assert int(gr.__version__.split('.')[0]) == 5, 'Se requiere Gradio 5'


### Criterio de continuidad

Si ves mensajes rojos *solo durante pip* pero la celda siguiente importa `torch`, `transformers`, `sentence_transformers` y `gradio` sin error, puedes continuar: pip está informando incompatibilidades globales de componentes de Colab. Si se produce un **ImportError** o falla la generación, detén la ejecución para diagnosticarlo. No reinstales bibliotecas repetidamente.

## 2. Problema, objetivos y diseño
**Problema.** La consulta manual de boletines públicos puede ser lenta y los LLM pueden inventar detalles de CVE.  
**Pregunta de investigación.** ¿En qué medida un recuperador semántico frente a TF-IDF mejora la identificación de fichas CISA KEV, y permite una respuesta que cite la fuente?  
**Hipótesis (a comprobar).** La combinación de información recuperada, CVE explícito y abstención es más trazable que generar respuestas sin referencias.  
**Límites.** KEV no contiene CVSS completo: no se deben inventar puntuaciones, ni fechas posteriores al snapshot descargado. No es un escáner ni un sistema de remediación automática.

## 3. Dataset público — CISA Known Exploited Vulnerabilities
Se descarga directamente de `https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json`. Si la conexión falla, se detiene con un mensaje claro; **no** se sustituye por registros sintéticos confundibles con datos reales. Se guarda el catálogo usado para reproducibilidad.
Si el dominio oficial no responde, revisa conectividad; **no** se fabrican registros de CVE.

In [ ]:
KEV_URL='https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json'
r=requests.get(KEV_URL,timeout=50,headers={'User-Agent':'CyberRAG-educational/1.2'}); r.raise_for_status()
raw=r.json(); assert isinstance(raw.get('vulnerabilities'),list) and raw['vulnerabilities'],'Respuesta KEV inválida'
(OUT/'cisa_kev_snapshot.json').write_text(json.dumps(raw,ensure_ascii=False,indent=2),encoding='utf-8')
df=pd.DataFrame(raw['vulnerabilities'])
print('Versión catálogo:',raw.get('catalogVersion'),'Fecha publicación:',raw.get('dateReleased'),'Registros:',len(df))
display(df.head(3))

## 4. EDA y curación de datos
Se explora la distribución por proveedores, fechas y ausencia de datos. Los indicadores se calculan del catálogo descargado, no de supuestos externos.

In [ ]:
print('Columnas:',list(df.columns))
print('Duplicados CVE:',int(df['cveID'].duplicated().sum()))
eda=pd.DataFrame({'columna':df.columns, 'faltantes':df.isna().sum().values,'porcentaje_faltantes':(df.isna().mean()*100).round(1).values})
display(eda)
display(df['vendorProject'].value_counts().head(12).rename_axis('proveedor').reset_index(name='casos'))
fig,axes=plt.subplots(1,2,figsize=(13,4))
df['vendorProject'].value_counts().head(10).sort_values().plot.barh(ax=axes[0],title='Top 10 proveedores KEV')
fechas=pd.to_datetime(df['dateAdded'],errors='coerce'); fechas.dt.year.value_counts().sort_index().plot.bar(ax=axes[1],title='Vulnerabilidades por año de adición')
plt.tight_layout();plt.savefig(OUT/'eda_cisa_kev.png',dpi=150);plt.show()
print('Fechas sin analizar:',int(fechas.isna().sum()))

## 5. Documentos con trazabilidad y fragmentación
Cada ficha se convierte en texto etiquetado con su `cveID`. El campo `knownRansomwareCampaignUse` es una **etiqueta declarada por CISA**, no una comprobación de una organización. Los fragmentos heredan metadatos. Dos tamaños permiten estudiar el efecto del chunking.

In [ ]:
FIELDS=[('cveID','CVE'),('vendorProject','Proveedor'),('product','Producto'),('vulnerabilityName','Vulnerabilidad'),('dateAdded','Fecha de incorporación KEV'),('shortDescription','Descripción'),('requiredAction','Acción requerida por CISA'),('dueDate','Fecha límite indicada'),('knownRansomwareCampaignUse','Uso ransomware conocido'),('notes','Notas'),('cwes','CWE')]
def as_text(rec):
    return '\n'.join(f'{label}: {rec.get(key)}' for key,label in FIELDS if rec.get(key) is not None and str(rec.get(key)).strip() not in ('','nan'))
def make_chunks(data,chunk_size=650,overlap=80):
    assert overlap<chunk_size
    output=[]
    for _,row in data.iterrows():
        cv=row['cveID']; text=as_text(row.to_dict())
        start=0; i=0
        while start<len(text):
            end=min(start+chunk_size,len(text)); output.append({'id':f'{cv}_{i}','cve':cv,'text':text[start:end],'source':KEV_URL})
            if end==len(text): break
            start=end-overlap;i+=1
    return output
chunks=make_chunks(df,650)
print('Fichas:',len(df),'Fragmentos:',len(chunks))
display(pd.DataFrame(chunks).head(2))

## 6. Recuperación: TF-IDF y embeddings E5
El modelo `intfloat/multilingual-e5-small` sigue el formato de entrenamiento `query:` y `passage:`. Para reducir tiempo en Colab, el índice semántico experimental trabaja con **hasta 600 fichas** (muestra fija más las fichas recientes), y documenta explícitamente esta limitación. Para consultar todo KEV se usa el índice léxico completo; la comparación de técnicas se hace sobre el **mismo subconjunto**.

In [ ]:
# Subconjunto reproducible (la evaluación se realiza solo sobre registros incluidos)
N_SEM=min(600,len(df))
subset=df.sample(n=N_SEM,random_state=SEED).reset_index(drop=True)
exp_chunks=make_chunks(subset,650)
texts=[x['text'] for x in exp_chunks]
vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=1,sublinear_tf=True)
X=vectorizer.fit_transform(texts)
EMBED_NAME='intfloat/multilingual-e5-small'
embedder=SentenceTransformer(EMBED_NAME,device='cuda' if torch.cuda.is_available() else 'cpu')
t0=time.perf_counter()
emb=embedder.encode(['passage: '+t for t in texts],batch_size=48,normalize_embeddings=True,show_progress_bar=True,convert_to_numpy=True)
print('Tiempo embeddings:',round(time.perf_counter()-t0,2),'s','Dimensiones:',emb.shape)

def search(query,k=3,method='semantic'):
    start=time.perf_counter()
    k=max(1,min(int(k),len(exp_chunks)))
    if method=='tfidf':
        scores=(X @ vectorizer.transform([query]).T).toarray().ravel()
    elif method=='semantic':
        q=embedder.encode(['query: '+query],normalize_embeddings=True,convert_to_numpy=True)[0]
        scores=emb@q
    else: raise ValueError('Método desconocido')
    ix=np.argsort(-scores)[:k]
    docs=[{**exp_chunks[j],'score':float(scores[j])} for j in ix]
    return docs,time.perf_counter()-start

prueba=search('¿Qué vulnerabilidad afecta a productos de Microsoft?',3,'semantic')[0]
display(pd.DataFrame([{k:d[k] for k in ('cve','score','source')} for d in prueba]))

## 7. Experimentos reproducibles — Hit@K y MRR@K
Las consultas de evaluación usan el identificador exacto de la ficha para construir un *benchmark diagnóstico de recuperación*. **Esto favorece las coincidencias literales** y no equivale a un benchmark de preguntas abiertas. Se reporta esta limitación explícitamente. Se prueban Top-K=3/5, TF-IDF/E5 y chunk 450/850, con igual muestra. La indexación de variantes puede tardar varios minutos.

In [ ]:
gold=subset.sample(n=min(30,len(subset)),random_state=21).copy()
queries=[(f'¿Qué acción requiere CISA para {r.cveID}?',r.cveID) for r in gold.itertuples()]
results=[]
for chunk_size in (450,850):
    docs=make_chunks(subset,chunk_size,80)
    corpus=[d['text'] for d in docs]
    v=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True).fit(corpus)
    mat=v.transform(corpus)
    dense=embedder.encode(['passage: '+t for t in corpus],batch_size=48,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=False)
    for method in ('tfidf','semantic'):
        for k in (3,5):
            hits=[]; reciprocal=[]; lat=[]
            for q,correct in queries:
                t0=time.perf_counter()
                if method=='tfidf': ss=(mat@v.transform([q]).T).toarray().ravel()
                else: ss=dense@embedder.encode(['query: '+q],normalize_embeddings=True,convert_to_numpy=True)[0]
                top=np.argsort(-ss)[:k]; retrieved=[docs[j]['cve'] for j in top]
                lat.append(time.perf_counter()-t0)
                hits.append(int(correct in retrieved))
                reciprocal.append(1/(retrieved.index(correct)+1) if correct in retrieved else 0)
            results.append({'chunk_caracteres':chunk_size,'metodo':method,'top_k':k,'Hit@K':np.mean(hits),'MRR@K':np.mean(reciprocal),'latencia_media_s':np.mean(lat),'n_preguntas':len(queries)})
metricas=pd.DataFrame(results)
display(metricas.round(4));metricas.to_csv(OUT/'metricas_retrieval.csv',index=False)
fig,ax=plt.subplots(figsize=(10,4));labels=metricas.apply(lambda x:f"{x.metodo} / {x.chunk_caracteres} / K{x.top_k}",axis=1)
ax.bar(range(len(labels)),metricas['Hit@K']);ax.set_xticks(range(len(labels)),labels,rotation=55,ha='right');ax.set_ylim(0,1.03);ax.set_ylabel('Hit@K');ax.set_title('Recuperación de CVE: evaluación diagnóstica');plt.tight_layout();plt.savefig(OUT/'comparacion_retrieval.png',dpi=150);plt.show()

## 8. Motor generativo Qwen 0.5B, sin Ollama
Predeterminado: `Qwen/Qwen2.5-0.5B-Instruct` para una demostración ligera. Se puede cambiar por `Qwen/Qwen2.5-1.5B-Instruct` en T4 si el tiempo y VRAM lo permiten. **No se comparan sus resultados directamente** con el modelo GGUF de Ollama sin controlar cuantización y condiciones.

In [ ]:
GEN_MODEL='Qwen/Qwen2.5-0.5B-Instruct'
t0=time.perf_counter()
tokenizer=AutoTokenizer.from_pretrained(GEN_MODEL)
llm=AutoModelForCausalLM.from_pretrained(GEN_MODEL,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,device_map='auto')
llm.eval();print('Motor:',GEN_MODEL,'Dispositivo:',llm.device,'Carga s:',round(time.perf_counter()-t0,2))

def generate_answer(question,contexts,max_new_tokens=140):
    if contexts:
        evidence='\n\n'.join(f"[Fuente {i+1} — {d['cve']}]\n{d['text']}" for i,d in enumerate(contexts))
        system='Eres un asistente técnico de ciberseguridad. Responde en español SOLO a partir del contexto suministrado. Si faltan datos, dilo. Nunca inventes CVSS, exploits ni medidas. Cita los identificadores CVE incluidos en el contexto.'
        prompt=f'CONTEXTO DOCUMENTAL:\n{evidence}\n\nPREGUNTA: {question}'
    else:
        system='Responde en español de manera breve. Esta es una prueba SIN documentos, no afirme haber consultado CISA.'
        prompt=question
    messages=[{'role':'system','content':system},{'role':'user','content':prompt}]
    inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt').to(llm.device)
    # Truncado conservador, dejando espacio para generación; informa el caso.
    if inputs.shape[1]>1600:
        raise ValueError('Contexto demasiado largo. Reduce Top-K o tamaño de fragmentos.')
    t0=time.perf_counter()
    with torch.inference_mode():
        out=llm.generate(inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    reply=tokenizer.decode(out[0][inputs.shape[-1]:],skip_special_tokens=True)
    return reply,time.perf_counter()-t0

# Prueba breve y verificable antes de levantar Gradio
print(generate_answer('Contesta solamente OK.',[],max_new_tokens=12))

## 9. Chatbot RAG con referencias y medición de latencia
Si se menciona un CVE, se busca primero en el subconjunto indexado; un CVE que **no** esté incluido se identifica como fuera de cobertura del índice experimental, sin reemplazarlo por un CVE similar. Las citas remiten al registro de CISA y al ID CVE. El enlace compartido por Gradio es público y temporal: evita cargar datos internos.

In [ ]:
INDEX_CVES={x['cve'] for x in exp_chunks}
LOG=[]
def chat_fn(message,history,top_k,metodo,usar_rag):
    start=time.perf_counter()
    cv=re.findall(r'CVE-\d{4}-\d{4,}',message.upper())
    if usar_rag and cv and cv[0] not in INDEX_CVES:
        return f'El identificador **{cv[0]}** no está en las {N_SEM} fichas del índice experimental. No se generará una recomendación basándose en otro CVE. Consulta el catálogo oficial: {KEV_URL}'
    docs=[]; trec=0
    if usar_rag:
        docs,trec=search(message,top_k,metodo)
        if cv:
            chosen=[d for d in exp_chunks if d['cve']==cv[0]]
            if chosen: docs=(chosen[:1]+[d for d in docs if d['cve']!=cv[0]])[:int(top_k)]
    try: response,tgen=generate_answer(message,docs)
    except Exception as e: return f'Fallo al generar: {type(e).__name__}: {e}'
    if docs:
        citations='\n'.join(f"- [{d['cve']}]({KEV_URL}) — fragmento {d['id']} (similitud {d.get('score',0):.3f})" for d in docs)
        response+='\n\n**Fuentes recuperadas (no equivalen a verificación de cada afirmación):**\n'+citations
    total=time.perf_counter()-start
    LOG.append({'pregunta':message,'rag':usar_rag,'metodo':metodo,'top_k':top_k,'recuperacion_s':round(trec,3),'generacion_s':round(tgen,3),'total_s':round(total,3),'respuesta':response})
    pd.DataFrame(LOG).to_csv(OUT/'interacciones_chat.csv',index=False)
    return response+f'\n\n_Tiempos: recuperación {trec:.2f} s · generación {tgen:.2f} s · total {total:.2f} s._'

with gr.Blocks(title='CyberRAG V1.3 — CISA KEV') as demo:
    gr.Markdown('# CyberRAG V1.3 | Asistente de vulnerabilidades CISA KEV\nConsulta educativa sobre un índice experimental de hasta 600 fichas. Respuestas con documentos trazables. No usar información confidencial.')
    with gr.Row():
        top_k=gr.Slider(1,5,value=3,step=1,label='Top-K')
        method=gr.Dropdown(['semantic','tfidf'],value='semantic',label='Recuperador')
        rag=gr.Checkbox(value=True,label='Usar contexto RAG')
    gr.ChatInterface(fn=chat_fn,additional_inputs=[top_k,method,rag],examples=[['¿Qué recomendaciones incluye CISA para las vulnerabilidades de Microsoft?',3,'semantic',True],['¿Qué datos aporta el catálogo KEV?',3,'tfidf',True]])
# Se imprime el link público para grabar el video. Ejecutar solo UNA instancia.
demo.queue(default_concurrency_limit=1).launch(share=True,debug=False)

## 10. Resultados, discusión y limitaciones (completar con evidencias reales)
Al terminar las consultas, descarga la carpeta de resultados. Incluye `metricas_retrieval.csv`, gráficas y `interacciones_chat.csv`. **No** se debe afirmar que RAG supera a un LLM sin RAG hasta evaluar respuestas con una rúbrica de fidelidad y fuentes.  
**Rúbrica manual sugerida:** respuesta basada en documento (0/1), correspondencia del CVE (0/1), abstención apropiada (0/1) y presencia de fuente (0/1). Comparar al menos cinco preguntas con y sin contexto.  
**Limitaciones:** muestra semántica de 600 KEV, preguntas de evaluación con CVE literal, modelo de 0.5B, inferencia dependiente de GPU e información que evoluciona. **El catálogo KEV no ofrece CVSS de manera uniforme**.

In [ ]:
print('Archivos generados:')
for f in OUT.iterdir(): print(' -',f.name,round(f.stat().st_size/1024,1),'KB')
# Ejecutar al finalizar para descargar evidencias
from google.colab import files
import shutil
archivo=shutil.make_archive('/content/CyberRAG_V1_2_resultados','zip',OUT)
print('ZIP de resultados:',archivo)
# Descomenta esta línea cuando desees descargar:
# files.download(archivo)